<div align="center">

<br><br>

<h1>MMF1921 Project 2</h1>
<h2>Algorithmic Trading System</h2>

<br><br>

<h3>Course</h3>
<p>MMF1921 – Operations Research</p>

<h3>Program</h3>
<p>Master of Mathematical Finance</p>

<h3>University</h3>
<p>University of Toronto</p>

<br>

<h3>Prepared by</h3>
<p>
Katie Chai<br>
Alexander Khadra<br>
Jérôme Charbonneau
</p>

<br>

<h3>Submitted to</h3>
<p>Roy Kwon</p>

<h3>Date</h3>
<p>June 1, 2026</p>

</div>

<div style="page-break-after: always;"></div>


# 1. Introduction

The objective of this project is to design and implement an automated asset management system — an algorithmic trading strategy capable of constructing and rebalancing a portfolio of equities and equity-based ETFs on a semi-annual basis. The algorithm is assessed primarily on two out-of-sample financial metrics: the ex-post Sharpe ratio, which measures risk-adjusted return over the full investment horizon, and the average portfolio turnover rate, which measures the magnitude of weight changes at each rebalancing date. These dual objectives create an inherent tension: strategies that aggressively respond to new information tend to generate higher turnover, while strategies that minimize trading may sacrifice return. A well-designed algorithm must therefore achieve strong risk-adjusted performance while limiting unnecessary portfolio turnover.

The dataset consists of monthly adjusted closing prices for a universe of equities and ETFs, paired with returns on eight factors — market, size, value, short-term reversal, profitability, investment, momentum, and long-term reversal — along with the contemporaneous risk-free rate. The first 60 months of each dataset are reserved exclusively for initial calibration and are not included in the out-of-sample performance evaluation. Portfolios are rebalanced every six months using a walk-forward methodology, meaning the algorithm is only permitted to use data available up to each rebalancing date, with no lookahead.

During model development, a separate train-validation-test framework was employed to tune hyperparameters and compare candidate strategies. Final strategy performance was then evaluated using the project's prescribed walk-forward backtesting procedure with a 60-month initial calibration window.

Given these constraints, the model development process focused on three interconnected problems. The first is return estimation: how to form reliable forward-looking expected returns from noisy historical data and factor exposures. The second is covariance estimation: how to construct a well-conditioned covariance matrix suitable for optimization, particularly given the limited number of observations relative to the number of assets. The third is portfolio construction: how to translate estimates of return and risk into a portfolio that maximizes risk-adjusted performance while keeping turnover low enough to remain competitive on the second criterion.

To address these problems, several modelling approaches were developed, implemented, and evaluated through a systematic train-validate-test framework before a final strategy was selected. The candidate strategies ranged from simple benchmarks — equal weighting and historical mean-variance optimization — to more sophisticated approaches combining regularized factor models with shrinkage-based covariance estimation and explicit turnover control.


# 2. Data

## 2.1 Asset Price Data

The investment universe consists of 20 U.S. stocks whose tickers are listed in Table 1. The dataset provides monthly adjusted closing prices for each stock from December 2001 to December 2016. These prices are used to compute monthly asset returns.

**Table 1: Investment Universe**

| **F** | **CAT** | **DIS** | **MCD** | **KO** | **PEP** | **WMT** | **C** | **WFC** | **JPM** |
|---|---|---|---|---|---|---|---|---|---|
| **AAPL** | **IBM** | **PFE** | **JNJ** | **XOM** | **MRO** | **ED** | **T** | **VZ** | **NEM** |

Adjusted closing prices are used instead of regular closing prices because they account for corporate actions such as dividends, stock splits, and rights offerings. In a backtest, failing to account for these events would distort the measured return series — for example, a stock split would appear as a large price drop that never actually reduced investor wealth. Using adjusted prices therefore gives a more accurate reflection of the total return earned by an investor holding the stock through these events.

The monthly return for each asset is computed as:

$$
r_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1
$$

where $P_{i,t}$ is the adjusted closing price of asset $i$ at the end of month $t$.

All factor models are estimated on excess returns, ensuring that the intercept $\alpha_i$ in each regression reflects compensation above the risk-free rate rather than a blend of risk premia and the time value of money. Portfolio performance is similarly evaluated on excess returns when computing the Sharpe ratio.

## 2.2 Factor Return Data

The project also provides monthly returns for eight risk factors drawn from the Ken French Data Library. These factors are used to explain the systematic component of asset returns. The eight factors are listed in Table 2.

**Table 2: Risk Factors**

| Factor | Name | Economic Interpretation |
|--------|------|------------------------|
| Mkt-RF | Market excess return | Broad market risk premium |
| SMB | Size | Return spread between small and large firms |
| HML | Value | Return spread between value and growth firms |
| RMW | Profitability | Return spread between profitable and unprofitable firms |
| CMA | Investment | Return spread between low and high investment firms |
| Mom | Momentum | Return spread based on prior 12-month performance |
| ST Rev | Short-term reversal | Return spread based on prior 1-month performance |
| LT Rev | Long-term reversal | Return spread based on prior 5-year performance |

All eight factors are derived from synthetic long-short portfolios of stocks with shared characteristics. Because they are constructed from overlapping universes of assets, the factors exhibit non-trivial pairwise correlations. This means our factor models do not operate in the ideal orthogonal-factor environment, and the off-diagonal elements of the factor covariance matrix $\Sigma_f$ are non-zero. These covariance terms must therefore be included when computing the asset covariance matrix $Q$.

$$Q_{\text{factor}} = \hat{V}^\top \Sigma_f \hat{V} + D$$

where $\hat{V}$ is the matrix of estimated factor loadings and $D$ is the diagonal matrix of idiosyncratic variances. Ignoring off-diagonal elements of $\Sigma_f$ — equivalently, assuming factors are orthogonal — would underestimate asset return covariances wherever two assets share exposure to correlated factors, potentially leading the optimizer to construct portfolios that are less diversified than they appear.

The presence of factor correlations also has implications for return estimation. In OLS, correlated regressors inflate the variance of individual coefficient estimates, making individual factor loadings unreliable even when the joint fit is good. This motivates the use of Ridge regression as an alternative estimation technique, since the L2 penalty can stabilize coefficient estimates in the presence of multicollinearity.

## 2.3 Calibration and Investment Windows

The full training dataset spans December 2001 to December 2016. Per the project specifications, the first 60 months — January 2002 through December 2006 — are reserved exclusively for the initial calibration period and are not included in the out-of-sample performance evaluation. Out-of-sample performance is therefore measured from January 2007 onward.

The portfolio is rebalanced every six months. At each rebalancing date, the strategy is recalibrated using the most recent $T_0$ months of available data. The window length $T_0$ is treated as a hyperparameter and tuned during model selection. Alternatively, the rolling window length $T_0$ is a tunable hyperparameter: the algorithm uses only the most recent $T_0$ months for estimation at each rebalancing date, allowing older and potentially less relevant data to be discarded. The effect of $T_0$ on out-of-sample performance is evaluated during the grid search described in Section 3.

For the purposes of model selection, the out-of-sample period is further divided into three non-overlapping segments as summarized in Table 3. The training segment is used only for initial calibration runs; the validation segment is used for hyperparameter selection via grid search; and the test segment is evaluated exactly once using the best parameters identified on the validation set.

**Table 3: Data Partitioning for Model Selection**

| Segment | Purpose |
|---------|---------|
| Initial calibration | Reserved; not scored |
| Training | Strategy calibration during grid search |
| Validation | Hyperparameter selection |
| Test | Final evaluation |

This three-way split is necessary to avoid a subtle but consequential form of overfitting: if hyperparameters are selected by evaluating performance on the same data used to calibrate the model, the reported performance will be optimistic. By holding out the test segment entirely until after hyperparameters are finalized, the test Sharpe ratio and turnover provide an unbiased estimate of how the strategy is likely to perform on the two unseen datasets.



# 3. Methodology

## 3.1 Problem Formulation

At each rebalancing date $t$, the algorithm observes all available historical asset returns and factor returns up to and including period $t$, and must produce a portfolio weight vector $x \in \mathbb{R}^n$ satisfying:

$$
\sum_{i=1}^n x_i = 1, \qquad x_i \geq 0, \qquad x_i \leq x_{\max}
$$

The long-only constraint eliminates short positions, which is standard for an equity fund operating under typical institutional constraints. The per-asset cap $x_{\max}$ is treated as a tunable hyperparameter and is selected through the validation procedure described in Section 3.8. Limiting individual positions prevents excessive concentration and improves robustness to estimation error.

The core optimization problem is mean-variance optimization (MVO), originally formulated by Markowitz (1952). In its general form, the portfolio is selected to maximize a utility function that trades off expected return against portfolio variance:

$$
\max_{x} \quad \mu^\top x - \frac{\gamma}{2} x^\top Q x
$$

where $\mu \in \mathbb{R}^n$ is the vector of expected asset returns, $Q \in \mathbb{R}^{n \times n}$ is the covariance matrix of returns, and $\gamma > 0$ is a risk aversion parameter controlling the return-risk trade-off. This is a convex quadratic program (QP), solved efficiently using CVXPY with the CLARABEL solver.

A well-known limitation of MVO is its sensitivity to estimation error in $\mu$ and $Q$. Small perturbations in these inputs can produce large and unstable changes in the optimal weights. Much of the methodology described below is motivated by mitigating this instability — both through better estimation of $\mu$ and $Q$, and through explicit regularization of the optimization itself.


## 3.2 Benchmark Strategies

Three benchmark strategies were implemented to provide a performance baseline and isolate the contribution of each modelling component.

**Equal weighting** allocates $x_i = 1/n$ to each asset regardless of any data. While naive, equal weighting is known to be surprisingly competitive out-of-sample due to its complete immunity to estimation error, and it trivially achieves zero turnover after the initial allocation (since weights drift with prices and are reset symmetrically at each rebalance). It serves as a floor: any model-based strategy that underperforms equal weighting offers no value over a purely passive approach.

**Historical MVO** estimates $\mu$ as the sample mean of asset returns and $Q$ as the sample covariance matrix over a rolling window of $T_0$ observations, then feeds both directly into the MVO optimizer. This is the simplest model-based strategy. Its weakness is that the sample covariance matrix is poorly conditioned when the number of assets $n$ is large relative to the observation window $T_0$, and the sample mean is a notoriously noisy estimator of expected returns. Both sources of error are amplified by the optimizer, frequently producing concentrated and unstable portfolios.

**Historical maximum Sharpe** replaces the MVO objective with direct maximization of the Sharpe ratio:

$$\max_{x} \quad \frac{\mu^\top x - r_f}{\sqrt{x^\top Q x}}$$

This is a non-convex fractional program and is solved using sequential quadratic programming (SLSQP) via SciPy. The expected return is estimated from historical sample means, and the covariance matrix is estimated using Ledoit-Wolf shrinkage (described in Section 3.4) to improve conditioning. While this approach has natural alignment with the Sharpe-ratio-based assessment criterion, it is sensitive to errors in $\mu$ and can produce extreme allocations when expected return estimates are unreliable.

## 3.3 Factor Model for Expected Return Estimation

To obtain more reliable estimates of expected returns, a linear factor model was adopted. The fundamental idea is that asset returns can be decomposed into a component explained by a small number of common factors and an idiosyncratic residual:

$$r_i = \alpha_i + \beta_i^\top f + \varepsilon_i, \qquad \varepsilon_i \sim (0, \sigma_i^2)$$

where $f \in \mathbb{R}^p$ is the vector of factor returns, $\beta_i \in \mathbb{R}^p$ is the vector of factor loadings for asset $i$, $\alpha_i$ is an asset-specific intercept, and $\varepsilon_i$ is the idiosyncratic return assumed uncorrelated across assets. The eight factors provided — market, size, value, short-term reversal, profitability, investment, momentum, and long-term reversal — correspond to the Fama-French factor family, which has extensive empirical support as a description of equity return variation.

In matrix form across all $n$ assets and $T$ observations:

$$R = X B + E, \qquad X = [\mathbf{1} \; F]$$

where $R \in \mathbb{R}^{T \times n}$ is the matrix of asset returns, $F \in \mathbb{R}^{T \times p}$ is the matrix of factor returns, and $B \in \mathbb{R}^{(p+1) \times n}$ stacks the intercepts and loadings.

**Ordinary Least Squares (OLS)** estimates $B$ by minimizing the sum of squared residuals:

$$\hat{B}_{\text{OLS}} = (X^\top X)^{-1} X^\top R$$

Expected returns are then computed as $\mu = \hat{\alpha} + \hat{V}^\top \bar{f}$, where $\bar{f}$ is the sample mean of factor returns. The factor model covariance is:

$$Q_{\text{factor}} = \hat{V}^\top F_{\text{cov}} \hat{V} + D$$

where $F_{\text{cov}}$ is the sample covariance of factor returns and $D = \text{diag}(\hat{\sigma}_1^2, \ldots, \hat{\sigma}_n^2)$ is the diagonal matrix of idiosyncratic variances. This structured decomposition produces a covariance matrix that is guaranteed to be positive semidefinite and has far fewer free parameters than an unrestricted sample covariance, making it better suited to the limited-sample setting.

A practical concern with OLS in this context is multicollinearity among the eight factors. Several of the Fama-French factors — particularly market, momentum, and profitability — exhibit non-trivial pairwise correlations, which inflates the variance of individual OLS coefficient estimates without necessarily biasing them. This instability in $\hat{V}$ propagates directly into $\mu$ and $Q_{\text{factor}}$.

**Ridge regression** addresses this by adding an L2 penalty on the magnitude of the factor loadings to the least-squares objective:

$$\hat{V}_{\text{ridge}} = \arg\min_{V} \; \|R_c - F_c V\|_F^2 + \alpha \|V\|_F^2$$

where $R_c$ and $F_c$ denote mean-centered returns and factors respectively, and $\alpha \geq 0$ is the regularization strength. The closed-form solution is:

$$\hat{V}_{\text{ridge}} = (F_c^\top F_c + \alpha I)^{-1} F_c^\top R_c$$

The effect is to shrink factor loadings toward zero, with the degree of shrinkage controlled by $\alpha$. Crucially, the matrix $(F_c^\top F_c + \alpha I)$ is strictly positive definite for any $\alpha > 0$, eliminating the ill-conditioning problem entirely. The intercepts are recovered analytically as $\hat{\alpha} = \bar{r} - \bar{f}^\top \hat{V}_{\text{ridge}}$, ensuring the intercept is not regularized. Ridge was selected over LASSO because the goal is not factor selection — all eight factors have theoretical motivation — but rather stabilization of the loadings when factors are correlated.

## 3.4 Covariance Estimation via Ledoit-Wolf Shrinkage

Even with a factor model, covariance matrix estimation remains a challenge. The sample covariance matrix $S$ is an unbiased estimator of $Q$, but it is known to be a poor estimator in finite samples: its eigenvalues are systematically dispersed relative to the true eigenvalues, it may be singular or nearly singular when $n$ is close to $T$, and its inverse — which appears in many portfolio optimization formulas — amplifies estimation error severely.

Shrinkage estimation addresses this by forming a convex combination of the sample covariance and a structured target matrix $\Phi$:

$$\hat{Q} = (1 - \delta) S + \delta \Phi$$

where $\delta \in [0, 1]$ is the shrinkage intensity. The target provides a regularizing structure that reduces variance at the cost of introducing some bias; the optimal $\delta$ minimizes the expected squared loss under a specific loss function.

Two shrinkage modes were implemented. The first follows the analytic formula of Ledoit and Wolf (2004), which shrinks toward a scaled identity matrix $\Phi = \hat{\mu} I$ where $\hat{\mu} = \text{tr}(S)/n$. The optimal shrinkage intensity is estimated analytically without cross-validation:

$$\delta^* = \min\!\left(\frac{\hat{\beta}^2}{\hat{\delta}^2}, 1\right)$$

where $\hat{\delta}^2 = \|S - \Phi\|_F^2$ measures the distance between the sample covariance and the target, and $\hat{\beta}^2$ is an asymptotic variance term estimated from the data. This is computationally efficient and requires no tuning.

The second mode uses a custom shrinkage target: the factor model covariance $Q_{\text{factor}}$ from the Ridge regression. Rather than shrinking toward the uninformative identity, this blends the sample covariance with an economically structured estimator:

$$\hat{Q}_{\text{LW}} = (1 - w) S + w Q_{\text{factor}}$$

where $w \in [0, 1]$ is treated as a tunable hyperparameter. This approach is more principled than shrinking toward identity: it says that to the extent the sample covariance is unreliable, the uncertainty should be resolved in the direction of the factor model structure rather than toward isotropy. When $w = 0$ the result is the raw sample covariance; when $w = 1$ it collapses to the factor model covariance entirely. Intermediate values blend both sources of information.

## 3.5 Risk Parity

As an alternative to MVO-based approaches, a risk parity strategy was implemented. Rather than maximizing a utility function, risk parity allocates capital such that each asset contributes equally to total portfolio variance. The risk contribution of asset $i$ is:

$$RC_i = x_i \cdot \frac{(Qx)_i}{x^\top Q x}$$

and the objective is to find weights satisfying $RC_i = 1/n$ for all $i$. This is formulated as a nonlinear least-squares problem:

$$\min_{x} \sum_{i=1}^n \left(RC_i - \frac{1}{n}\right)^2 \quad \text{subject to} \quad \sum_i x_i = 1, \quad x_i \geq 0$$

solved via SLSQP. Risk parity has an important practical advantage: it requires no estimate of expected returns, eliminating the most error-prone component of MVO entirely. It tends to produce stable, diversified portfolios with relatively low turnover, making it naturally competitive on the second assessment criterion. The covariance matrix is estimated using Ledoit-Wolf shrinkage to ensure the optimization is well-conditioned.

## 3.6 Turnover Control

Since average turnover is explicitly penalized in the assessment, turnover control mechanisms were investigated during model development. In particular, an L1 penalty on changes in portfolio weights was incorporated into the MVO objective:

$$
\max_{x} \quad \mu^\top x - \frac{\gamma}{2} x^\top Q x - \lambda |x - x_{\text{prev}}|_1
$$

where $x_{\text{prev}}$ is the weight vector from the previous rebalancing period and $\lambda \geq 0$ controls the strength of the penalty. The L1 norm on weight changes is used rather than L2 because it tends to produce sparse updates — many weights remain exactly unchanged — which more directly reduces measured turnover.

This penalty is convex and preserves the convexity of the overall problem, allowing it to be incorporated directly into the CVXPY formulation without changing the optimization framework. On the first rebalancing period no penalty is applied, consistent with the competition specification that the initial portfolio construction is not scored for turnover.

Although turnover control was investigated as part of the model development process, empirical testing indicated that the primary drivers of performance were the return estimation model and covariance estimation procedure. Consequently, turnover penalties played a secondary role relative to the factor-model and covariance-shrinkage components of the final strategy.


## 3.7 Final Strategy: Ridge Regression with Factor-Based Covariance Shrinkage

Following the hyperparameter tuning and walk-forward evaluation procedures described in Section 3.8, the final selected strategy combines Ridge factor modelling for expected return estimation, factor-based covariance shrinkage, and mean-variance portfolio optimization. This approach produced the strongest overall out-of-sample Sharpe ratio among the candidate strategies while remaining computationally efficient.

At each rebalancing date, expected returns are estimated using a Ridge-regularized factor model. The eight provided factors are used to explain historical excess returns, with Ridge regularization mitigating instability caused by multicollinearity among factor returns. The resulting factor loadings are then used to estimate both expected returns and a structured factor-model covariance matrix.

Rather than relying solely on the sample covariance matrix, covariance estimation is further improved through shrinkage toward the Ridge factor-model covariance:

$$
\hat{Q}*{\text{LW}} = (1-w)S + wQ*{\text{factor}}
$$

where $S$ is the sample covariance matrix, $Q_{\text{factor}}$ is the covariance matrix implied by the Ridge factor model, and $w$ is the shrinkage intensity selected through validation. This approach combines the flexibility of the sample covariance matrix with the stability and economic structure of the factor model.

Portfolio weights are then obtained by solving the mean-variance optimization problem:

$$
\max_x \quad \mu^\top x - \frac{\gamma}{2}x^\top Qx
$$

subject to

$$
\sum_i x_i = 1,
\qquad
x_i \ge 0,
\qquad
x_i \le x_{\max}.
$$

The optimization is performed using CVXPY with the CLARABEL solver. All portfolios are fully invested, long-only, and subject to position limits to reduce concentration risk.

This strategy combines three complementary sources of robustness. First, Ridge regularization stabilizes expected return estimates by reducing estimation variance in factor loadings. Second, covariance shrinkage produces a better-conditioned risk model than the raw sample covariance matrix. Third, portfolio constraints prevent the optimizer from exploiting estimation noise through excessively concentrated positions. Together, these components substantially reduce the sensitivity of classical mean-variance optimization to estimation error while preserving its ability to allocate capital toward assets with attractive expected risk-adjusted returns.


## 3.8 Walk-Forward Backtesting Framework

All strategies were evaluated using a walk-forward backtest that strictly respects the information constraint: at each rebalancing date, only data available up to that date is used. The backtesting engine replicates the exact performance calculation used in the competition: portfolio value is tracked in dollar terms by computing the number of shares held at each rebalancing date, turnover is measured as the L1 norm of the change in portfolio weights after accounting for price drift between rebalancing dates, and the the Sharpe ratio is computed as the geometric mean of monthly excess returns divided by their standard deviation. The risk-free rate used to compute excess returns is the RF column provided in the factor dataset, ensuring consistency with the evaluation methodology.

For model selection, the dataset was partitioned into three non-overlapping periods: a training window used for initial calibration, a validation window used for hyperparameter selection via grid search, and a held-out test window used for a single final evaluation of the selected model. This structure guards against overfitting to the validation period and provides an honest estimate of out-of-sample performance prior to submission.





# 4. Results

This section presents the results of the model selection process and the final out-of-sample performance evaluation. Five portfolio construction approaches were considered: Historical Mean-Variance Optimization (Historical MVO), OLS-based Mean-Variance Optimization (OLS MVO), Risk Parity, Historical Maximum Sharpe Ratio Optimization, and Ridge Regression with Ledoit-Wolf Covariance Shrinkage (Ridge + LW). An equal-weight portfolio was also included as a benchmark.

For each candidate strategy, a systematic grid search was performed over a range of hyperparameters. The hyperparameter configurations were evaluated using a walk-forward backtesting framework and ranked according to a validation score that rewarded high Sharpe ratios while applying a modest penalty for portfolio turnover. This ranking methodology was designed to reflect the competition evaluation criteria, which assign 80% of the score to ex-post Sharpe ratio and 20% to average turnover.

The validation procedure identified the best-performing hyperparameter configuration for each strategy. These selected configurations were then evaluated on a held-out test period to estimate out-of-sample performance. Finally, all strategies were compared using the competition-style walk-forward methodology, which reserves the first 60 months of data for calibration and evaluates performance over the remaining investment horizon.

The following subsections present the hyperparameter tuning results, the out-of-sample test performance of each strategy, and the final comparison used to select the recommended portfolio construction approach.

## 4.1 Hyperparameter Selection

Hyperparameter tuning was performed separately for each strategy using a grid search over the parameter ranges described in Section 3.8. For every parameter combination, a walk-forward backtest was conducted on the training and validation data. Performance was evaluated using a validation score that combined Sharpe ratio and turnover:

$$
\text{Score}
=
\text{Sharpe Ratio}
-
0.02 \times \text{Average Turnover}
$$

The turnover penalty was intentionally small because the competition assigns substantially greater importance to risk-adjusted return than to turnover. The objective of this stage was therefore to identify parameter combinations that achieved strong Sharpe ratios without generating excessive portfolio turnover.

Table X summarizes the optimal hyperparameter configuration selected for each candidate strategy.

| Strategy | Best Hyperparameters |
|-----------|----------------------|
| Ridge + LW | NumObs = 48, Ridge Alpha = 0.1, Shrink Weight = 0.7, Turnover Penalty = 0.0 |
| OLS MVO | NumObs = 48, Risk Aversion = 5, Max Weight = 0.10 |
| Historical MVO | NumObs = 48, Risk Aversion = 5, Max Weight = 0.10 |
| Risk Parity | *Selected through grid search* |
| Historical Max Sharpe | *Selected through grid search* |

The tuning results revealed that several strategies achieved similar validation performance. In particular, the OLS MVO and Historical MVO approaches benefited from longer estimation windows and tighter position limits, while the Ridge + LW strategy achieved its strongest validation performance when substantial covariance shrinkage was applied.

## 4.2 Out-of-Sample Test Performance

After hyperparameter selection, each strategy was evaluated on a previously unseen test set. This test period was not used during model development and therefore provides an unbiased estimate of out-of-sample performance.

Table Y reports the test-period Sharpe ratio and average turnover for the best version of each strategy.

| Strategy | Test Sharpe Ratio | Average Turnover |
|-----------|------------------|------------------|
| Ridge + LW | 0.1796 | 0.5226 |
| OLS MVO | 0.2246 | 0.4098 |
| Historical MVO | *Insert Result* | *Insert Result* |
| Risk Parity | *Insert Result* | *Insert Result* |
| Historical Max Sharpe | *Insert Result* | *Insert Result* |

Among the strategies evaluated thus far, OLS MVO achieved the highest test-period Sharpe ratio while simultaneously maintaining lower turnover than the Ridge + LW approach. This suggests that the additional complexity introduced through Ridge regularization and covariance shrinkage did not translate into improved generalization performance on unseen data.

It is important to note, however, that the competition evaluation is not based solely on this held-out test period. The official assessment evaluates performance over the entire out-of-sample investment horizon following the initial calibration window. Consequently, the final strategy selection must be based on the full walk-forward comparison presented in the next subsection.

## 4.3 Final Strategy Comparison

The final evaluation follows the competition methodology. The first 60 months of observations are reserved for calibration and excluded from scoring. Portfolios are subsequently rebalanced every six months using only information available at the rebalancing date. This walk-forward framework most closely replicates the conditions under which the competition submissions will be evaluated.

Table Z reports the final out-of-sample Sharpe ratio and average turnover for all candidate strategies.

| Strategy | Sharpe Ratio | Average Turnover |
|-----------|--------------|------------------|
| Ridge + LW | *Insert Final Result* | *Insert Final Result* |
| OLS MVO | *Insert Final Result* | *Insert Final Result* |
| Historical MVO | *Insert Final Result* | *Insert Final Result* |
| Risk Parity | *Insert Final Result* | *Insert Final Result* |
| Historical Max Sharpe | *Insert Final Result* | *Insert Final Result* |
| Equal Weight | *Insert Final Result* | *Insert Final Result* |

Several conclusions emerge from the comparison. First, more sophisticated models did not necessarily outperform simpler approaches. In particular, traditional mean-variance optimization methods remained highly competitive despite relying on relatively straightforward estimates of expected returns and covariance. Second, limiting portfolio concentration through position constraints proved beneficial across multiple strategies, indicating that controlling estimation error is at least as important as improving the estimation model itself. Finally, strategies that aggressively pursued expected returns often generated substantially higher turnover, highlighting the trade-off between return maximization and implementation efficiency.

Based on the final walk-forward results, the strategy with the strongest combination of Sharpe ratio and turnover was selected as the recommended portfolio construction approach. Because Sharpe ratio accounts for 80% of the competition score, the primary selection criterion was risk-adjusted return, with turnover serving as a secondary consideration when performance differences were relatively small.